# 🌌 Level 5: High-Dimensional Regression & The Curse of Dimensionality

**[📖 Want a detailed explanation? Read the Manual (Streamlit App)](https://bookseal-seoul-apt-price-prediction.streamlit.app/Level_5_High_Dimensional)**

We started with 1 feature (Area), then 2, then 3.
What if we add **100 features**?

**Hypothesis**: More info = Better model?
**Reality**: Not always! Let's see how "Garbage Features" confuse a model.

### 💡 Mental Model: The Needle in a Giant Room

Imagine dropping a needle (the pattern) in a small room (Low Dimension). It's easy to find.

Now imagine dropping it in a **football stadium** (High Dimension). It's incredibly hard to find.

Adding more dimensions (features) spreads your data out thinly, making it harder for the model to find the true pattern.

### 1. Load Data
Standard setup.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

url = "https://github.com/bookseal/seoul-apt-price-prediction/raw/main/data/sample.parquet"
df = pd.read_parquet(url)
print(f"Data loaded: {len(df):,} rows")

### 2. Experiment: The Feature Spammer
We will try to predict Price using **Area** + **Random Noise**.
The noise contains ZERO useful information. A smart human would ignore it.
What does Linear Regression do?

In [ ]:
# Setup experiment
X_real = df[['area_m2']].values
y = df['price_10k_krw'].values

noise_levels = [0, 10, 50, 100, 200, 500]
results = []

for n_noise in noise_levels:
    # 1. Generate random noise
    np.random.seed(42)
    if n_noise > 0:
        noise = np.random.rand(len(df), n_noise)
        X = np.hstack([X_real, noise])
    else:
        X = X_real
        
    # 2. Split
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    
    # 3. Train
    model = LinearRegression()
    model.fit(X_train, y_train)
    
    # 4. Evaluate
    train_rmse = np.sqrt(mean_squared_error(y_train, model.predict(X_train)))
    test_rmse = np.sqrt(mean_squared_error(y_test, model.predict(X_test)))
    
    results.append({
        'Noise Features': n_noise,
        'Train RMSE': train_rmse,
        'Test RMSE': test_rmse
    })

# Show results
res_df = pd.DataFrame(results)
res_df

### 3. Visualizing Overfitting

In [ ]:
plt.figure(figsize=(10, 6))
plt.plot(res_df['Noise Features'], res_df['Train RMSE'], 'o-', label='Train Error (Memorization)')
plt.plot(res_df['Noise Features'], res_df['Test RMSE'], 'o-', label='Test Error (Real Accuracy)', color='red')

plt.xlabel('Number of Noise Features added')
plt.ylabel('RMSE (Error)')
plt.title('The Curse of Dimensionality: Overfitting')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

### 4. Conclusion
1. **Train RMSE drops**: The model "memorizes" the noise to fit the training data perfectly.
2. **Test RMSE explodes**: The model fails completely on new data because it learned fake patterns.

**Lesson**: Just adding more data isn't always good! We need to select the *right* features.